In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# B: fixed BASE aggregate-payload / state alignment diagnostic

Run all measures exactly two independent VJPs at the same saved JOINT BASE,
or resumes unfinished units in the same Drive task. The fixed payload scalar
is the mean of message-aligned margins at historical indices
**[0,1,4,5,7,9,12,13,16,17,18,20,27,29,30,31]**. The fixed state scalar is
**correlation[0]-correlation[16]**. A fresh readout may have a different
nonpositive roster or active wrong offset; the notebook records that difference
without redefining a scalar or rejecting the observation.

The BASE and historical CONT_FINAL tensors come from the existing Six-Bit-Final
`run/inputs/fixed_inputs.pt` fields `base` and `final`. Old and residual-compensated
bridge directions use the completed Terminal-Bridge `tensors/raw.pt` and
`tensors/reconstruction_residual.pt`. These are the actual saved large-tensor
locations used by the earlier runtimes. Paths and SOURCE_REF are ordinary editable
locators; source/version identity is not an execution gate.

Each scalar uses a fresh decoder graph through the proven checkpoint/replay disk
spool path. Every forward saves q, all 32 margins, all 22 correlations, fresh G,
fixed-scalar value, active offsets, and the fresh nonpositive roster. Each complete
unit then atomically saves the raw gradient, masked gradient, both norms, outside-mask
norm, replay ledger and call receipts. Zero, adverse, or conflicting gradients are
complete results; there is no sign, margin, protection or scientific-success gate.

CPU analysis projects the two unnormalized masked gradients onto three fixed unit
directions: `M(raw)`, `M(raw-reconstruction_residual)`, and `CONT_FINAL-BASE`.
It saves masked direction norms, pairwise unit-direction cosines, and the 2x3 raw
scalar slopes, gradient norms and dimensionless gradient/direction cosines, plus the
two-gradient Gram and cosine for payload/state competition. Direction
normalization defines unit displacement only; gradients are never normalized or
relabeled as another scalar unit. A zero gradient has zero slopes and null cosines;
a zero saved direction has no unit direction, so its slope and cosine are null.

Nominal work is **1 VAE load + 2 gradient decodes + 2 VJPs**, 92 forward and
92 nominal recompute chunks, 176 windows, 2,816 chips and 110 metrics at one
source point. Ordinary decode, encode, DiT, native/scheduler, codec and candidate
generation are all zero. No scan, update, 17-VJP extension or stop/restart step.
Units run serially and independently.

Stable task: `MyDrive/Video-WM/Local-Joint-State-Payload-V1-Base-Alignment/`
`resumable-base-alignment-v1/`. Every 60 seconds saves last committed progress.
Each gradient commits immediately. Repeated Run all skips complete units and a
fully complete task needs no VAE/CUDA. An interrupted unfinished unit restarts as
one unit; attempts, failure history and observed physical replay counts accumulate.

Completion means the fixed observations and projection report were saved. This
known-message one-point diagnostic cannot establish per-bit common improvement,
which individual bit is sacrificed, protection of the other 16 bits, ROI-specific
gradient conflict, finite-step branch switching, blind recovery, codec survival,
generalization or scientific PASS. GPU/Colab execution is user-run. Return the
entire stable task directory.

In [ ]:
from pathlib import Path
import datetime, json, os, signal, subprocess, sys, time, traceback
SOURCE_REF = 'dev/local-joint-state-payload-v1'  # Editable GitHub branch/tag/ref.
FIXED_CONFIG = {'schema': 'local-joint-base-alignment-v1', 'source_run': 'BASE plus saved terminal-bridge and CONT_FINAL tensors', 'source_arm': 'JOINT_BASE', 'model': {'id': 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers', 'revision': '0fad780a534b6463e45facd96134c9f345acfa5b'}, 'carrier': {'key': 'local-joint-state-payload-v1-first-mechanism', 'message_hex': '8001a55a', 'rho': 0.5, 'cap': 1.0}, 'nonpositive_roster': [0, 1, 4, 5, 7, 9, 12, 13, 16, 17, 18, 20, 27, 29, 30, 31], 'fixed_wrong_offset': 16, 'device': 'cuda', 'inputs': {'six_bit_inputs': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Six-Bit-Final/resumable-six-bit-final-v1/run/inputs/fixed_inputs.pt', 'bridge_run': '/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Terminal-Bridge/20261010T031603868487Z/run'}}
EXPECTED_COST = {'vae_load': 1, 'decode': 2, 'decoder_vjp': 2, 'ordinary_decode': 0, 'encode': 0, 'dit': 0, 'native': 0, 'scheduler': 0, 'codec': 0, 'ordinary_chunks': 0, 'gradient_forward_chunks': 92, 'nominal_replay_chunks': 92, 'windows': 176, 'chips': 2816, 'metrics': 110, 'independent_source_points': 1}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/Local-Joint-State-Payload-V1-Base-Alignment')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / 'resumable-base-alignment-v1'  # Same logical task on every Run all.
OUTPUT.mkdir(exist_ok=True)
ATTEMPT_OUTPUT = OUTPUT / 'notebook_attempts' / STAMP
ATTEMPT_OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = OUTPUT / 'run'
WORKSPACE = Path('/content/Video-WM-Local-Joint-Base-Alignment-' + STAMP)
PYTHON = sys.executable
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

def original_record_failure(stage, exc):
    row = dict(stage=stage, reason=f'{type(exc).__name__}: {exc}', traceback=''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)), retry=False, output=str(OUTPUT), expected_cost=EXPECTED_COST)
    write_json(ATTEMPT_OUTPUT / 'notebook_failure.json', row)

def logged(command, stage, *, cwd=None, env=None, check=True):
    child = None
    returncode = None
    primary = None
    primary_tb = None
    cleanup_errors = []

    def cleanup_error(label, exc):
        cleanup_errors.append(label + ': ' + repr(exc))
    try:
        with (ATTEMPT_OUTPUT / 'execution.log').open('a', encoding='utf-8') as log:
            line = 'STAGE ' + stage + ' COMMAND ' + repr(command) + '\n'
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
            group = {'start_new_session': True} if os.name == 'posix' else {}
            child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **group)
            try:
                for line in child.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = child.wait()
            except BaseException as exc:
                primary = exc
                primary_tb = exc.__traceback__
            finally:
                if child is not None:
                    if os.name == 'posix':
                        try:
                            os.killpg(child.pid, signal.SIGTERM)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGTERM', exc)
                        deadline = time.monotonic() + 5.0
                        while time.monotonic() < deadline:
                            try:
                                os.killpg(child.pid, 0)
                            except ProcessLookupError:
                                break
                            except BaseException as exc:
                                cleanup_error('group probe', exc)
                                break
                            time.sleep(0.05)
                        try:
                            os.killpg(child.pid, signal.SIGKILL)
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('group SIGKILL', exc)
                    elif child.poll() is None:
                        try:
                            child.terminate()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('terminate', exc)
                    try:
                        returncode = child.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        try:
                            child.kill()
                        except ProcessLookupError:
                            pass
                        except BaseException as exc:
                            cleanup_error('kill', exc)
                        try:
                            returncode = child.wait(timeout=10)
                        except BaseException as exc:
                            cleanup_error('wait after kill', exc)
                    except BaseException as exc:
                        cleanup_error('wait', exc)
                    if child.stdout is not None:
                        try:
                            child.stdout.close()
                        except BaseException as exc:
                            cleanup_error('stdout close', exc)
            if primary is None and (not cleanup_errors):
                log.write('EXIT ' + str(returncode) + '\n')
                log.flush()
    except BaseException as exc:
        if primary is None:
            primary = exc
            primary_tb = exc.__traceback__
        else:
            cleanup_error('log wrapper', exc)
    if primary is None and cleanup_errors:
        primary = RuntimeError('process cleanup failed: ' + '; '.join(cleanup_errors))
        primary_tb = primary.__traceback__
    if primary is None and check and returncode:
        primary = subprocess.CalledProcessError(returncode, command)
        primary_tb = primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary, 'add_note'):
            primary.add_note('secondary cleanup errors: ' + '; '.join(cleanup_errors))
        try:
            record_failure(stage, primary)
        except BaseException as record_error:
            if hasattr(primary, 'add_note'):
                primary.add_note('failure record error: ' + repr(record_error))
        raise primary.with_traceback(primary_tb)
    return returncode

def record_failure(stage, exc):
    original_record_failure(stage, exc)
    fixed_slots.update(status='ENGINEERING_FAILURE', reason=repr(exc))
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
    if (RUN_OUTPUT / 'result.json').exists():
        from runtime.wan.local_joint_base_alignment_v1 import finalize_interrupted
        finalize_interrupted(RUN_OUTPUT, 'notebook takeover after child cleanup: ' + repr(exc))

fixed_slots = dict(status='SETUP_STARTED', expected_cost=EXPECTED_COST,
    units={name: dict(status='NOT_RUN', expected_windows=88, expected_chips=1408, expected_metrics=55)
           for name in ('aggregate_payload', 'state_G16')})
write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
if (OUTPUT / 'fixed_config.json').exists():
    FIXED_CONFIG = json.loads((OUTPUT / 'fixed_config.json').read_text())
else:
    write_json(OUTPUT / 'fixed_config.json', FIXED_CONFIG)
write_json(ATTEMPT_OUTPUT / 'setup_receipt.json', dict(config=FIXED_CONFIG, expected_cost=EXPECTED_COST,
    output=str(OUTPUT), python=sys.version, scientific_pass=False, automatic_retry=False))
print('Fixed BASE-ALIGNMENT output:', OUTPUT, flush=True)

In [ ]:
import io, urllib.request, zipfile
SOURCE_URL = 'https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/' + SOURCE_REF + '/notebooks/local_joint_base_alignment_v1_portable_source.zip'
try:
    with urllib.request.urlopen(SOURCE_URL) as response:
        package_bytes = response.read()
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
        for member in archive.infolist():
            target = (WORKSPACE / member.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise RuntimeError('unsafe source archive path')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
    # A repeated Run all may retain packages from an earlier extracted workspace.
    import importlib
    for module_name in list(sys.modules):
        if module_name.split('.', 1)[0] in ('main', 'runtime', 'experiments'):
            sys.modules.pop(module_name, None)
    importlib.invalidate_caches()
    sys.path.insert(0, str(WORKSPACE))
    print('Source:', SOURCE_URL, 'Workspace:', WORKSPACE, flush=True)
    try:
        write_json(ATTEMPT_OUTPUT / 'source_receipt.json', dict(url=SOURCE_URL, workspace=str(WORKSPACE)))
    except Exception as exc:
        print('Source note unavailable:', repr(exc), flush=True)
except BaseException as exc:
    record_failure('SOURCE_DOWNLOAD', exc)
    raise

In [ ]:
from runtime.wan.local_joint_base_alignment_v1 import model_work_remaining
try:
    NEEDS_MODEL = model_work_remaining(RUN_OUTPUT)
    probe_path = OUTPUT / 'dependency_probe.json'
    probe_lines = [
        "import json, pathlib, sys, torch, numpy, PIL",
        "row = dict(python=sys.version, torch=str(torch.__version__), model_work_remaining=" + repr(NEEDS_MODEL) + ")",
    ]
    if NEEDS_MODEL:
        probe_lines += ["from diffusers import AutoencoderKLWan"]
        if str(FIXED_CONFIG.get('device', 'cuda')).startswith('cuda'):
            probe_lines += ["assert torch.cuda.is_available(), 'CUDA is needed for unfinished decode/VJP'",
                            "row.update(cuda_available=True, device=torch.cuda.get_device_name(0))"]
    probe_lines += ["pathlib.Path(" + repr(str(probe_path)) + ").write_text(json.dumps(row))"]
    probe_code = '\n'.join(probe_lines)
    probe_returncode = logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_PROBE', check=False)
    if probe_returncode:
        packages = ['torch', 'numpy', 'pillow']
        if NEEDS_MODEL:
            packages += ['diffusers', 'transformers', 'accelerate', 'safetensors', 'huggingface-hub', 'sentencepiece', 'ftfy']
        logged([PYTHON, '-m', 'pip', 'install', *packages], 'DEPENDENCY_REPAIR')
        logged([PYTHON, '-u', '-c', probe_code], 'DEPENDENCY_REPROBE')
    dependency_check_returncode = logged([PYTHON, '-m', 'pip', 'check'], 'DEPENDENCY_REPORT', check=False)
    environment_receipt = json.loads(probe_path.read_text())
    environment_receipt.update(dependency_check_returncode=dependency_check_returncode,
        source_and_versions_are_location_notes=True, checkpoint_interval_seconds=60)
    write_json(ATTEMPT_OUTPUT / 'environment_receipt.json', environment_receipt)
    print('Remaining model work:', NEEDS_MODEL, '; task:', OUTPUT, flush=True)
except BaseException as exc:
    record_failure('ENVIRONMENT_SETUP', exc)
    raise

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token: os.environ['HF_TOKEN'] = hf_token
except Exception as exc:
    print('Optional HF token unavailable:', type(exc).__name__, flush=True)

run_returncode = None
try:
    command = [PYTHON, '-u', '-m', 'experiments.wan_state_clock.local_joint_base_alignment_v1_run',
               '--config', str(OUTPUT / 'fixed_config.json'), '--output', str(RUN_OUTPUT)]
    run_returncode = logged(command, 'FIXED_BASE_ALIGNMENT', cwd=WORKSPACE, check=False)
    from runtime.wan.local_joint_base_alignment_v1 import finalize_interrupted
    # Takeover is report-only and follows child/process-group reap in logged().
    finalize_interrupted(RUN_OUTPUT, 'child exited with return code ' + str(run_returncode))
    result_path = RUN_OUTPUT / 'result.json'
    result = json.loads(result_path.read_text()) if result_path.exists() else None
    write_json(ATTEMPT_OUTPUT / 'execution_receipt.json', dict(returncode=run_returncode,
        status=None if result is None else result['status'],
        actual_counts=None if result is None else result['counts'], expected_cost=EXPECTED_COST,
        scientific_pass=False, automatic_retry=False))
    if run_returncode != 0 or result is None or result['status'] != 'COMPLETE':
        raise RuntimeError('BASE-ALIGNMENT engineering interruption; retain entire output: ' + str(OUTPUT))
    fixed_slots.update(status='SUPERSEDED_BY_RUN_RESULT', result='run/result.json')
    write_json(OUTPUT / 'fixed_slots.json', fixed_slots)
except BaseException as exc:
    try: record_failure('FIXED_BASE_ALIGNMENT', exc)
    except BaseException as record_error:
        if hasattr(exc, 'add_note'): exc.add_note('failure record error: ' + repr(record_error))
    raise

In [ ]:
result = json.loads((RUN_OUTPUT / 'result.json').read_text())
print('Engineering status:', result['status'])
print('Nominal budget:', result['expected_cost'])
print('Cumulative physical calls:', result['counts'])
print('This Run all:', result.get('current_attempt_counts', {}))
print('Fixed totals:', result.get('totals'))
for name, row in result['units'].items():
    readout = row.get('readout', {})
    print(name, row['status'], row.get('gradient_norms'),
          'fresh nonpositive=', readout.get('fresh_nonpositive_bits'),
          'fresh active wrong=', readout.get('fresh_active_wrong_offsets'),
          'reason=', row.get('reason'))
for name, replay in result['replay'].items():
    print('Replay:', name, replay['counts'], replay.get('boundary_storage', {}))
diagnostic_path = RUN_OUTPUT / 'diagnostic.json'
if diagnostic_path.exists():
    diagnostic = json.loads(diagnostic_path.read_text())
    print('Direction geometry:', diagnostic['direction_geometry'])
    print('Unit-direction cosines:', diagnostic['unit_direction_cosines'])
    print('Two-gradient geometry:', diagnostic['gradient_geometry'])
    print('2x3 projections:', diagnostic['projections'])
print(result['ceiling'])
print('Return the entire output folder, including failures:', OUTPUT)